### html to pdf RAG

##### **Turns out I don't want a merged pdf!** I want a merged markdown file:

- notebooklm immediately strips out images and stuff, from pdf, web pages whatever
- it also accepts markdown
- The readability library does a good job of cleaning html web pages, then converting to markdown
- The pymupdf4llm is good and cleaning pdfs and converting to markdown
- markdown files are smaller than pdf or html, so more likely to fit into notebooklm's data limits
- note that notebooklm may not be able to realize that a merged markdown file is really from many sources (I've got no solution to this)

##### Anyway, what this does is...

Does the  RAG html-to-pdf processing recommended [here](https://www.perplexity.ai/search/when-processing-html-files-for-bxWgyfVNQ4OUkzGp0a4d3g#0), although, it's modifed to put clickable links in the margin, as [here](https://www.perplexity.ai/search/modify-the-code-below-so-that-M48v941eRhW2dxy_y2AeWQ#1).

I tried this with weasyprint but there were problems I can't remember now.  I left this at using BeautifySoup and PdfMerger.

But maybe this is better `winget install --id=tschoonj.GTKForWindows -e`

##### WeasyPrint relies on the following libraries:
- Pango: For text layout and rendering.
- Cairo: For graphics rendering.
- GDK-PixBuf: For image handling.

You can get all of these by installing GTK3 from here: https://github.com/tschoonj/GTK-for-Windows-Runtime-Environment-Installer/releases
By default, this will be installed in  C:\Program Files\GTK3-Runtime Win64
Must add C:\Program Files\GTK3-Runtime Win64\bin to your path

*But I gave up on weaprint for this use*

##### Marging pdfs, either in memory or from separate files
From [here](https://www.perplexity.ai/search/in-the-code-below-a-single-htm-M6muR3A4RRC49NJEgq2yAw#0)
- **required libs** 
   conda install -c conda-forge pypdf2
   conda install -c conda-forge cchardet

#### GoogleLM limitations from [here](https://support.google.com/notebooklm/answer/14278184?hl=en)
- max of 200 MB total uploads (accross whole notebookLM?)
- 500k word limit per source (would determine how many articles can be lumped together)


In [ ]:
# ------------- conda setup test  -----------------------
# I see fontconfig warnings, but perplexity assures me I can ignore them.  Fix requires setting my path to a conda environment, which is bad.
# import numpy as np
# import pandas as pd
# from bs4 import BeautifulSoup
# from weasyprint import HTML

# print("Environment setup successful!")

In [2]:
import refwrangle as rfw
from icecream import ic
import pathlib as pl

In [5]:
from pathlib import Path
from bs4 import BeautifulSoup
import chardet
from PyPDF2 import PdfMerger

def clean_html(html_file_path):
    """Cleans an HTML file by removing unwanted elements."""
    # Detect the file encoding
    with open(html_file_path, 'rb') as file:
        raw_data = file.read()
        result = chardet.detect(raw_data)
        encoding = result['encoding']

    # Now read with detected encoding
    with open(html_file_path, 'r', encoding=encoding) as file:
        soup = BeautifulSoup(file, 'html.parser')

    # Remove unwanted elements like <script> and <style>
    for tag in soup(['script', 'style']):
        tag.decompose()

    # Return cleaned HTML as a string
    return str(soup)

def html_to_pdf_bytes(cleaned_html, basename, html_path):
    """Converts cleaned HTML content to PDF bytes."""
    from fpdf import FPDF

    pdf = FPDF()
    pdf.add_page()
    pdf.set_font("Arial", size=12)
    
    # Add a title or header for each HTML file
    pdf.cell(0, 10, f"Content from: {basename}", ln=True, align='C')
    
    # Add the cleaned HTML content to the PDF
    pdf.multi_cell(0, 10, cleaned_html)
    
    # Return PDF as bytes
    return pdf.output(dest='S').encode('latin1')

def process_multiple_htmls(html_paths, output_pdf_path):
    """Processes multiple HTML files and merges them into a single PDF."""
    merger = PdfMerger()

    for html_path in html_paths:
        ic(html_path)
        cleaned_html = clean_html(html_path)
        basename = html_path.stem
        pdf_bytes = html_to_pdf_bytes(cleaned_html, basename, html_path)

        # Save temporary PDF for merging
        temp_pdf_path = Path(f"{basename}.pdf")
        with open(temp_pdf_path, 'wb') as temp_pdf:
            temp_pdf.write(pdf_bytes)

        # Append to the merger
        merger.append(str(temp_pdf_path))

        # Remove temporary PDF after merging
        temp_pdf_path.unlink()

    # Write the merged PDF to the output path
    with open(output_pdf_path, 'wb') as output_pdf:
        merger.write(output_pdf)

# # Example usage
# htmlFNms = [Path("example1.html"), Path("example2.html")]  # Replace with your actual HTML file paths
# output_pdf_path = Path("./merged_htmls.pdf")
# process_multiple_htmls(htmlFNms, output_pdf_path)

In [6]:
htmlFNms = ['Tumulty24FrischLearnedDemsShould.html','Yan24berkeleyFuncCallLeaderBrd.html','Walther24barstoolConservatism.html']
htmlFNms = [rfw.lit_attachment_dir_shared / htmlFNm for htmlFNm in htmlFNms]
#htmlFNms

output_pdf_path = pl.Path("./merged_htmls.pdf")
process_multiple_htmls(htmlFNms, output_pdf_path)

ic| html_path: WindowsPath('C:/Users/scott/OneDrive/share/ref/obsidian/Obsidian Share Vault/lit/lit_sources/Tumulty24FrischLearnedDemsShould.html')


UnicodeEncodeError: 'latin-1' codec can't encode character '\u2019' in position 2217: ordinal not in range(256)

In [ ]:
# html_file_path = pl.Path(r"C:\Users\scott\OneDrive\share\ref\refwrangle\test\Dionne24hiddenVictoryProgrssiv.html")
# output_pdf_path = pl.Path("./tmp_textonly.pdf")
# process_html_to_pdf(html_file_path, output_pdf_path)

In [ ]:
# merging pdf files.  Saves memory but is slower.

# from PyPDF2 import PdfMerger
# merger = PdfMerger()
# for pdf in pdf_list:
#     merger.append(pdf)
# merger.write("output.pdf")
# merger.close()
# ```[3]

# 2. **PyMuPDF**:
# ```python
# import fitz
# result = fitz.open()
# for pdf in pdf_list:
#     with fitz.open(pdf) as mfile:
#         result.insert_pdf(mfile)
# result.save("output.pdf")
# ```[5]